# 18 — Vectorized Transformations, Series `.apply()`, and Element-Wise Mapping
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to writing high-throughput Pandas transformations, mastering `.map()` vs `.apply()`, and avoiding costly Python row-loop traps.*

---

## 📌 Executive Summary & Interview Expectations
In production data engineering and data science workflows, transformation logic often determines whether an automated pipeline completes in 2 seconds or 2 hours. Interviewers frequently test your understanding of **vectorization vs Python iteration**, specifically scrutinizing:
1. **Series `.apply()` vs Vectorized `.str` Accessors**: Why lambdas in `.apply()` bypass NumPy C-speed and incur massive Python interpreter overhead.
2. **DataFrame `.map()` vs Deprecated `.applymap()`**: Modern Pandas 2.1+ and 3.0 API changes and the unified mapping interface.
3. **Branching Logic**: Implementing conditional categorization using vectorized boolean arrays and `np.select` instead of `def func(x): if...else...` row-by-row functions.
4. **Row-wise (`axis=1`) Anti-Patterns**: Identifying the severe performance penalties of iterating across rows and learning the idiomatic alternatives.

## 1. Environment Setup & Data Ingestion

In [1]:
import numpy as np
import pandas as pd
import time

address = 'student-mat.csv'
df = pd.read_csv(address)

print(f"Dataset Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

Dataset Loaded Successfully: 395 rows, 33 columns


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10


## 2. Label-Based Slicing with `.loc`

### 💡 Interview Tip: Endpoint Inclusion in `.loc`
- `.iloc[start:stop]` follows standard Python zero-indexed half-open slice conventions (`stop` is **excluded**).
- `.loc[:, 'start_col':'end_col']` uses **closed intervals** (`end_col` is **included**).
- Here, slicing from `'school'` to `'guardian'` captures both boundary columns.

In [2]:
# Slice dataframe from 'school' through 'guardian'
sliced_df = df.loc[:, "school":"guardian"].copy()
print("Sliced DataFrame Shape:", sliced_df.shape)
sliced_df.head(4)

Sliced DataFrame Shape: (395, 12)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother


## 3. String Transformations: Vectorized `.str` vs `.apply()`

### ⚠️ Common Junior Pattern vs Senior Production Standard
```python
# Junior Approach (Python bytecode loop via lambda):
sliced_df['Mjob'] = sliced_df['Mjob'].apply(lambda x: x.capitalize())

# Senior Production Standard (Vectorized C-level execution):
sliced_df['Mjob'] = sliced_df['Mjob'].str.capitalize()
```

#### Why `.str` accessors dominate:
1. **Performance**: Vectorized string accessors run optimized C routines under the hood.
2. **Null Safety**: `.str` methods automatically propagate `NaN` values without raising `AttributeError: 'float' object has no attribute 'capitalize'`.
3. **Readability**: Expressive, idiomatic Pandas style.

In [3]:
# Compare both approaches for demonstration
capitalizer = lambda x: x.capitalize()

# Apply capitalization to Mjob and Fjob
sliced_df["Mjob"] = sliced_df["Mjob"].str.capitalize()
sliced_df["Fjob"] = sliced_df["Fjob"].str.capitalize()

print("Transformed Mjob and Fjob (Tail elements):")
sliced_df.tail(4)

Transformed Mjob and Fjob (Tail elements):


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian
391,MS,M,17,U,LE3,T,3,1,Services,Services,course,mother
392,MS,M,21,R,GT3,T,1,1,Other,Other,course,other
393,MS,M,18,R,LE3,T,3,2,Services,Other,course,mother
394,MS,M,19,U,LE3,T,1,1,Other,At_home,course,father


## 4. Boolean Column Creation: Vectorized Comparison vs Python Function

### ⚠️ The `apply(function)` Trap for Boolean Flags
Many tutorials write:
```python
def majority(x):
    if x > 17:
        return True
    return False
df['legal_drinker'] = df['age'].apply(majority)
```
In Python, this calls the `majority` function individually for every single row, jumping in and out of the Python interpreter stack.

**Vectorized Alternative**:
`df['legal_drinker'] = df['age'] > 17`
This executes as a single contiguous SIMD-accelerated array comparison in C/NumPy!

In [4]:
# Senior Vectorized Implementation
sliced_df["legal_drinker"] = sliced_df["age"] > 17

print("Sample with new 'legal_drinker' flag:")
sliced_df[["school", "sex", "age", "guardian", "legal_drinker"]].head(5)

Sample with new 'legal_drinker' flag:


,school,sex,age,guardian,legal_drinker
0,GP,F,18,mother,True
1,GP,F,17,father,False
2,GP,F,15,mother,False
3,GP,F,15,mother,False
4,GP,F,16,father,False


## 5. Element-Wise DataFrame Transformations

### 🚨 Crucial Pandas 2.x & 3.0 Modernization Note:
- `DataFrame.applymap()` was officially **deprecated in Pandas 2.1** and **removed in Pandas 3.0**.
- The unified method is now **`DataFrame.map()`** (matching `Series.map()`).
- In addition, if you want to scale all numeric columns by 10, vectorized arithmetic on `select_dtypes` is orders of magnitude faster than iterating cell-by-cell!

In [5]:
# 1. Element-wise function using modern DataFrame.map()
def ten_multiplier(x):
    if isinstance(x, (int, float)) and not isinstance(x, bool):
        return 10 * x
    return x

# Using modern .map() (replaces deprecated .applymap())
mapped_df = sliced_df.map(ten_multiplier)
print("Result of element-wise DataFrame.map(ten_multiplier):")
display(mapped_df.head(4))

Result of element-wise DataFrame.map(ten_multiplier):


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,legal_drinker
0,GP,F,180,U,GT3,A,40,40,At_home,Teacher,course,mother,True
1,GP,F,170,U,GT3,T,10,10,At_home,Other,course,father,False
2,GP,F,150,U,LE3,T,10,10,At_home,Other,other,mother,False
3,GP,F,150,U,GT3,T,40,20,Health,Services,home,mother,False


In [6]:
# 2. Production Alternative: Vectorized In-Place Numeric Scaling
# Instead of inspecting every cell's type, scale all numeric columns in one shot:
numeric_scaled = sliced_df.copy()
numeric_cols = numeric_scaled.select_dtypes(include=['int64', 'float64']).columns
numeric_scaled[numeric_cols] = numeric_scaled[numeric_cols] * 10

print("Result of vectorized numeric scaling (select_dtypes):")
display(numeric_scaled.head(4))

Result of vectorized numeric scaling (select_dtypes):


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,legal_drinker
0,GP,F,180,U,GT3,A,40,40,At_home,Teacher,course,mother,True
1,GP,F,170,U,GT3,T,10,10,At_home,Other,course,father,False
2,GP,F,150,U,LE3,T,10,10,At_home,Other,other,mother,False
3,GP,F,150,U,GT3,T,40,20,Health,Services,home,mother,False


## 6. The Pandas Vectorization Performance Hierarchy

| Rank | Method | Engine | Relative Speed | When to Use |
| :--- | :--- | :--- | :--- | :--- |
| **1 (Fastest)** | Vectorized Operations (`df['A'] * 10`, `df['A'] > 17`) | NumPy / C SIMD | **1x (Baseline, ~100x-1000x faster)** | Arithmetic, boolean masks, standard comparisons |
| **2** | Vectorized Accessors (`.str`, `.dt`) | Cython / C | **~5x - 10x slower than 1** | Text manipulation, date extractions |
| **3** | `np.where()` / `np.select()` | NumPy C | **~1x - 3x slower than 1** | Complex multi-branch conditionals |
| **4** | `Series.map(dict)` | Hash Table lookup | **~10x slower than 1** | Categorical re-mapping and dictionaries |
| **5** | List Comprehensions `[f(x) for x in df['A']]` | Optimized Python loop | **~20x - 50x slower than 1** | Custom string parsing when `.str` lacks the feature |
| **6** | `Series.apply(fn)` | Python bytecode loop | **~50x - 100x slower than 1** | Arbitrary Python objects |
| **7 (Slowest)** | `DataFrame.apply(fn, axis=1)` | Row-by-row Series creation | **~500x - 2000x slower than 1** | **Avoid in production pipelines!** |

---
## 🎯 7. Technical Interview Corner: Tricky Questions & Drills

### Q1: What is the exact distinction between `Series.map()`, `Series.apply()`, `DataFrame.apply()`, and `DataFrame.map()`?
**Answer**:
1. **`Series.map(arg)`**: Maps values of a Series using an input correspondence (dict, Series, or function). Best for key-value dictionary substitution.
2. **`Series.apply(func)`**: Applies an arbitrary Python function across every element of the Series.
3. **`DataFrame.apply(func, axis=0/1)`**: Applies an aggregation or transformation function along an entire axis.
   - `axis=0` (default): passes each **column** as a Series to `func`.
   - `axis=1`: passes each **row** as a Series to `func` (very slow!).
4. **`DataFrame.map(func)`**: Applies a function element-wise across **every individual cell** in the 2D DataFrame (replaced `applymap`).

---

### Q2: Why is `df.apply(axis=1)` so notoriously slow?
**Answer**:
For every single row in the DataFrame, Pandas must:
1. Construct a new `pd.Series` object with the column names as index.
2. Box the data into Python objects.
3. Call the Python function and handle the return value.
4. Reassemble the results into a DataFrame/Series.
For a DataFrame with 100,000 rows, this creates 100,000 temporary Series objects!

---

### Q3: Advanced Interview Coding Drill: Multi-Condition Risk Flagging
**Challenge**: Compute a `Risk_Profile` for each student based on weekly alcohol consumption (`Walc`) and grade 3 performance (`G3`):
- `'High Risk'`: `Walc >= 4` AND `G3 < 10`
- `'Moderate Risk'`: `Walc >= 3` OR `G3 < 12`
- `'Low Risk'`: All other students

*Requirement*: Implement this using both **`np.select()`** (production vectorized standard) and measure the output distribution.

In [7]:
# Interview Solution: High-Performance Vectorized Conditional Branching with np.select
conditions = [
    (df["Walc"] >= 4) & (df["G3"] < 10),
    (df["Walc"] >= 3) | (df["G3"] < 12)
]
choices = ["High Risk", "Moderate Risk"]

df_analyzed = df.assign(
    Risk_Profile=np.select(conditions, choices, default="Low Risk")
)

print("Distribution of Risk Profiles across Students:")
display(df_analyzed["Risk_Profile"].value_counts(dropna=False))

print("\nSample students with their computed Risk Profile:")
display(df_analyzed[["school", "age", "Walc", "absences", "G1", "G2", "G3", "Risk_Profile"]].head(5))

Distribution of Risk Profiles across Students:


Risk_Profile
Moderate Risk    259
Low Risk         106
High Risk         30
Name: count, dtype: int64


Sample students with their computed Risk Profile:


,school,age,Walc,absences,G1,G2,G3,Risk_Profile
0,GP,18,1,6,5,6,6,Moderate Risk
1,GP,17,1,4,5,5,6,Moderate Risk
2,GP,15,3,10,7,8,10,Moderate Risk
3,GP,15,1,2,15,14,15,Low Risk
4,GP,16,2,4,6,10,10,Moderate Risk
